In [0]:
# 1. Variables du compte de stockage
storage_account_name = "stairqualitydevnu81f1"

# 2. Récupération dynamique de la clé via le Secret Scope
storage_account_key = dbutils.secrets.get(scope="kv-scope", key="stairqualitydevnu81f1-key")

# 3. Chemin du fichier JSON dans le conteneur Bronze
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/openaq/raw_air_quality.json"

# 4. Lecture du fichier JSON en passant la clé dans les options
df_bronze = (
    spark.read
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
    .option("multiline", "true")
    .json(bronze_path)
)

# 5. Affichage du schéma et des données
df_bronze.printSchema()
display(df_bronze)

In [0]:
from pyspark.sql.functions import explode, col, coalesce, lit, to_timestamp

# 1. Chemin de destination dans le conteneur Silver
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/openaq/air_quality_cleaned"

# 2. Pipeline de transformation et nettoyage Silver
df_silver = (
    df_bronze
    # Aplatissement du tableau 'results'
    .select(explode(col("results")).alias("loc"))
    
    # Sélection, renommage et typage explicite
    .select(
        col("loc.id").cast("long").alias("location_id"),
        col("loc.name").alias("location_name"),
        # Remplacement des valeurs NULL par une valeur par défaut
        coalesce(col("loc.locality"), lit("Unspecified")).alias("locality"),
        col("loc.country.code").alias("country_code"),
        col("loc.coordinates.latitude").cast("double").alias("latitude"),
        col("loc.coordinates.longitude").cast("double").alias("longitude"),
        # Conversion du texte en vrai format Timestamp
        to_timestamp(col("loc.datetimeFirst.utc")).alias("first_updated_at"),
        to_timestamp(col("loc.datetimeLast.utc")).alias("last_updated_at")
    )
    
    # Filtre de qualité : suppression des lignes sans coordonnées GPS
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    
    # Dédoublonnage sur la clé primaire
    .dropDuplicates(["location_id"])
)

# 3. Affichage du résultat final
display(df_silver)

# 4. Sauvegarde au format Delta Lake (avec écrasement du schéma existant)
(
    df_silver.write
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
    .mode("overwrite")
    .option("overwriteSchema", "true")  # <-- Autorise le changement de structure de la table
    .format("delta")
    .save(silver_path)
)

print("✅ Couche Silver nettoyée et sauvegardée avec succès en Delta Lake !")